In [1]:
# Import required libraries
import pandas as pd
from pathlib import Path
import glob, re, html, quopri

# Define the data directory (relative to the notebook path)
# Notebook is in: phishing-detector/notebooks/
DATA_DIR = Path("../data")

In [2]:
# Imports used by the cleaners
import re, html, quopri

# Collapse only the punctuation spacing that commonly breaks URL matching.
def normalize_punct_spacing_for_urls(s):
    s = str(s)
    s = re.sub(r'\s*\.\s*', '.', s)    # remove spaces around '.'
    s = re.sub(r'\s*/\s*', '/', s)     # remove spaces around '/'
    s = re.sub(r'\s*:\s*', ':', s)     # remove spaces around ':'
    s = re.sub(r'(?i)\b(https?|ftp|file)\s*[\.:;]\s*/\s*/', r'\1://', s)  # https.//, http;// -> https://, http://
    return re.sub(r'\s+', ' ', s).strip()

# Regex for URLs and protocol-like tokens
URL_RE = re.compile(
    r"(?:"
    r"((https?|ftp|file)\s*:[^\s]*)"                                   # http:, https:, ftp:, file:
    r"|"
    r"(www\.[^\s]+)"                                                   # www.example.com
    r"|"
    r"(\b[A-Za-z0-9.-]+\.(com|net|org|info|biz|ru|cn|io|co)\b[^\s]*)"  # bare domain
    r"|"
    r"(\bhttps?\b)"                                                    # bare 'http' or 'https'
    r"|"
    r"(\bwww\s+[A-Za-z0-9.-]+\s+\b(com|net|org|info|biz|ru|cn|io|co)\b)"# 'www anywheremd com'
    r")",
    flags=re.IGNORECASE,
)

# Minimal patterns to strip HTML and mail/MIME header noise that leaks into bodies
HTML_TAG_RE = re.compile(r"<[^>]+>")
HEADER_RE = re.compile(
    r"^(from|to|cc|bcc|subject|date|reply-to|return-path|message-id|received|"
    r"x-[\w-]+|mime-version|content-type|content-transfer-encoding|boundary)\s*:.*$",
    flags=re.IGNORECASE | re.MULTILINE,
)
BOUNDARY_RE = re.compile(r"^--[-_=A-Za-z0-9]+$", flags=re.MULTILINE)

# Clean up the URL and other headers
def clean_text_for_bert(text):
    text = str(text)
    
    # Decode quoted-printable artifacts like '=\r\n', '=20'
    try:
        text = quopri.decodestring(text).decode("utf-8", "ignore")
    except Exception:
        pass

    # Make common URL obfuscations detectable by the regex
    text = normalize_punct_spacing_for_urls(text)

    # Drop mail headers and MIME boundary lines
    text = HEADER_RE.sub("", text)
    text = BOUNDARY_RE.sub("", text)

    # Remove raw HTML tags
    text = HTML_TAG_RE.sub(" ", text)

    # Replace anything that looks like a URL/protocol/bare domain with a neutral token
    text = URL_RE.sub(" [URL] ", text)

    # Unescape entities (&amp; -> &, etc.), normalize spaces, and lowercase
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

In [3]:
# Create a DataFrame list
dfs = []

# Files to load from ../data
files = [
    "CEAS_08.csv",
    "Enron.csv",
    "Ling.csv",
    "Nazario.csv",
    "Nazario_5.csv",
    "SpamAssassin.csv",
    "TREC_05.csv",
    "TREC_06.csv",
    "TREC_07.csv",
    "Nigerian_Fraud.csv",
    "Nigerian_5.csv",
]

# Global read options: use utf-8, ignore bad bytes, avoid dtype warnings
READ_KWARGS = {"encoding": "utf-8", "encoding_errors": "ignore", "low_memory": False}

# Load each dataset, clean, and append
for name in files:
    path = f"../data/{name}"
    print(f"Loading {name} ...")

    # Load CSV
    df = pd.read_csv(path, **READ_KWARGS)

    # Standardize column names
    df.columns = [c.strip().lower() for c in df.columns]

    # Ensure subject/body columns exist (allow either to be empty)
    if "subject" not in df.columns:
        df["subject"] = ""
    if "body" not in df.columns:
        df["body"] = ""

    # Build text = subject + body
    df["text"] = (
        df["subject"].astype(str).fillna("") + " " +
        df["body"].astype(str).fillna("")
    ).str.strip()

    # Clean text (remove URLs, HTML, and headers)
    df["text"] = df["text"].apply(clean_text_for_bert)

    # Require a label column
    if "label" not in df.columns:
        print(f"  Skipped {name}: no 'label' column")
        continue

    # Drop rows with NaN label
    df = df.dropna(subset=["label"])

    # Keep only rows where label is strictly 0 or 1 (numeric or string)
    mask_label_allowed = df["label"].isin([0, 1]) | df["label"].astype(str).str.strip().isin(["0", "1"])
    df = df[mask_label_allowed]

    # Convert all "0" and "1" strings to integers
    df["label"] = df["label"].astype(int)

    # Drop rows where text is empty after cleaning
    df = df[df["text"] != ""]

    # Keep only text and label
    df = df[["text", "label"]]

    # Add to the DataFrame list
    print(f"  kept {len(df)} rows")
    dfs.append(df)

# Combine all datasets
df_all = pd.concat(dfs, ignore_index=True)

Loading CEAS_08.csv ...
  kept 39033 rows
Loading Enron.csv ...
  kept 29725 rows
Loading Ling.csv ...
  kept 2859 rows
Loading Nazario.csv ...
  kept 1563 rows
Loading Nazario_5.csv ...
  kept 3046 rows
Loading SpamAssassin.csv ...
  kept 5780 rows
Loading TREC_05.csv ...
  kept 54948 rows
Loading TREC_06.csv ...
  kept 16339 rows
Loading TREC_07.csv ...
  kept 53670 rows
Loading Nigerian_Fraud.csv ...
  kept 3230 rows
Loading Nigerian_5.csv ...
  kept 6206 rows


In [4]:
# Combine all DataFrames into one
df_all = pd.concat(dfs, ignore_index=True)

# Remove duplicate rows based on text
before = len(df_all)
df_all = df_all.drop_duplicates(subset=["text"])
after = len(df_all)
print(f"Removed {before - after} duplicate entries")

Removed 16662 duplicate entries


In [5]:
# Shuffle rows and reset index
df_all = df_all.sample(frac=1, random_state=2025).reset_index(drop=True)
print("After cleaning:", df_all.shape)

After cleaning: (199737, 2)


In [6]:
# Display dataset shape and label distribution
print("Shape:", df_all.shape)
print("Label distribution:\n", df_all["label"].value_counts())

# Show the first few rows
df_all.head()

Shape: (199737, 2)
Label distribution:
 label
0    107533
1     92204
Name: count, dtype: int64


,text,label
0,re:uribl > 71.920 77.1604 1.2331 0.984 0.71 0....,0
1,can you last 36 hours ? mol ci - ialis softabs...,1
2,"home delivery cials-tabs ""the beverage.icee, b...",1
3,fw:presentation tammie schoppe enron americas-...,0
4,good day.what's so good about it?:) [url] grab...,1


In [7]:
# Create the data folder if it does not exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save the cleaned and combined dataset into the data folder
output_path = DATA_DIR / "combined.csv"
df_all.to_csv(output_path, index=False)

print(f"Saved combined dataset to: {output_path.resolve()}")

Saved combined dataset to: /home/nmd/projects/phishing-detector/data/combined.csv
